In [ ]:
!pip install -U bnlp_toolkit

In [ ]:
from datasets import load_dataset
ds = load_dataset("ai4bharat/samanantar", "bn")

In [ ]:
corpus = ds.data['train'].to_pandas()

In [ ]:
df = corpus.head(100)

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from bnlp import BengaliPOS
import random

# --- Step 2: Initialize Models ---
embedding_model = SentenceTransformer("sentence-transformers/LaBSE")
bn_pos = BengaliPOS()

# --- Step 3: Define Replacement Rules via POS Tags ---
REPLACEABLE_TAGS = {
    'NC',  # Common Noun
    'NP',  # Proper Noun
    'NV',  # Verbal Noun
    'JJ',  # Adjective
    'VM',  # Main Verb (use cautiously)
    'JQ',  # Quantifier (e.g., "many" → "অনেক")
}

def get_replaceable_words(sentence_bn):
    """Extract words with replaceable POS tags."""
    tagged = bn_pos.tag(sentence_bn)
    return [
        (word, pos) for word, pos in tagged
        if pos in REPLACEABLE_TAGS
    ]

# --- Step 4: Context-Aware Code-Mixing ---
def code_mix_with_pos(sentence_bn, sentence_en, threshold=0.75, replace_prob=1):
    # Step 1: Get replaceable Bengali words and their POS
    replaceable_words = get_replaceable_words(sentence_bn)
    if not replaceable_words:
        return sentence_bn

    bn_words, bn_pos_tags = zip(*replaceable_words)

    # Step 2: Tokenize English translation (clean and lowercase)
    en_words = [
        word.lower() for word in sentence_en.split()
        if word.isalpha()
    ]
    if not en_words:
        return sentence_bn

    # Step 3: Compute embeddings
    bn_embeddings = embedding_model.encode(bn_words, convert_to_tensor=True)
    en_embeddings = embedding_model.encode(en_words, convert_to_tensor=True)

    # Step 4: Similarity matrix
    sim_matrix = cosine_similarity(bn_embeddings.cpu(), en_embeddings.cpu())

    # Step 5: Replace words with POS-based rules
    tokens = sentence_bn.split()
    replaced_tokens = []
    for token in tokens:
        if token in bn_words and random.random() < replace_prob:
            idx = bn_words.index(token)
            best_en_idx = np.argmax(sim_matrix[idx])
            best_similarity = sim_matrix[idx][best_en_idx]

            # POS-specific replacement logic
            pos = bn_pos_tags[idx]
            if (
                (best_similarity > threshold) and
                (pos in REPLACEABLE_TAGS) # and
                # Skip replacing verbs if English candidate isn't a verb (optional)
                # not (pos == 'VM' and not en_words[best_en_idx].endswith(('ing', 'ed')))
            ):
                replaced_tokens.append(en_words[best_en_idx])
            else:
                replaced_tokens.append(token)
        else:
            replaced_tokens.append(token)

    return " ".join(replaced_tokens)

# --- Step 5: Apply to DataFrame ---
df["code_mixed"] = df.apply(
    lambda row: code_mix_with_pos(row["tgt"], row["src"]),
    axis=1
)

# Save results
df.to_csv("code_mixed_pos_aware.csv", index=False)